# 02 · Reading benchmark results

How to read the output of `scripts/benchmark_skill.py` (or `jobs/benchmark.sbatch`). Set `TABLE` to your run's `benchmark_table.csv`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / 'fishcast').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

TABLE = ROOT / 'artifacts' / 'benchmark' / 'benchmark_table.csv'  # change me
t = pd.read_csv(TABLE)
t[['model', 'track', 'skill_vs_constant', 'skill_ci_low', 'skill_ci_high', 'pr_auc', 'pr_auc_vs_no_skill']].round(3)

## The columns

| Column | Meaning |
| --- | --- |
| `track` | `climate` = environment, location and season only (usable for 2040). `nowcast` = also uses recent krill catches in the same area. `baseline` = the constant prediction |
| `skill_vs_constant` | How much smaller the model's error is than always predicting the training average. 0.05 = 5% better; negative = worse |
| `skill_ci_low/high` | 95% interval from resampling whole survey years. If it includes 0, we can't say the model beats the average |
| `pr_auc` | How well the model ranks real hotspots above other hauls (0 to 1). Random guessing scores the hotspot share, about 0.10 |
| `pr_auc_vs_no_skill` | `pr_auc` divided by the random score. Our bar for a forecast claim is at least 2 |

## The skill check (semester plan, section 7)

A climate-only model passes if its density skill interval is above zero **and** its PR-AUC is at least twice random.

In [ ]:
climate = t[t.track == 'climate'].copy()
climate['passes'] = (climate.skill_ci_low > 0) & (climate.pr_auc_vs_no_skill >= 2)
climate[['model', 'skill_vs_constant', 'skill_ci_low', 'pr_auc_vs_no_skill', 'passes']].round(3)

## Chart

The same figure the Data Viz team uses:

In [ ]:
import subprocess
out = ROOT / 'viz' / 'figures' / 'out' / 'leaderboard'
subprocess.run([sys.executable, str(ROOT / 'viz' / 'figures' / 'leaderboard.py'), str(TABLE), '--out', str(out), '--hide-seeds'], check=True)
from IPython.display import Image
Image(str(out) + '.png')